<a href="https://colab.research.google.com/github/deepthivj-aiml/5-Day-AI-Agents-Intensive-course-Google/blob/main/Copy_of_Fork_of_AutoEvalAI_AI_Powered_Second_Hand_Vehicle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade gradio opencv-python-headless numpy pandas requests nest-asyncio google-genai google-adk
!npm install -g @modelcontextprotocol/server-everything

  Using cached gradio-6.0.2-py3-none-any.whl.metadata (16 kB)
  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached google_genai-1.53.0-py3-none-any.whl.metadata (47 kB)
  Using cached google_adk-1.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached gradio_client-2.0.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached websockets-15.0.1-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached gradio-6.0.2-py3-none-any.whl (21.6 MB)
Using cached gradio_client-2.0.1-py3-none-any.whl (55 kB)
Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.5 MB)
Using cached google_genai-1.53.0-py3-none-any.whl (262 kB)
Using cached google_adk-1.20.0-py3-none-any.whl (2.3 MB)
Using cached websockets-15.0.1-cp312-cp312-manylinux_2_

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇^C


In [ ]:
# ==============================================
# 🔧 Install dependencies
# ==============================================
!pip install -q pillow gradio google-generativeai

# ==============================================
# 🔑 Imports
# ==============================================
import google.generativeai as genai
from PIL import Image, ImageEnhance, ImageFilter
import gradio as gr
import io, time, re

# ==============================================
# 🔑 Setup Gemini
# ==============================================
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-2.0-flash")

# ==============================================
# ✨ Image Enhancement
# ==============================================
def enhance(img):
    img = img.convert("RGB")
    img = img.filter(ImageFilter.SHARPEN)
    img = ImageEnhance.Contrast(img).enhance(1.25)
    img = ImageEnhance.Color(img).enhance(1.15)
    img = ImageEnhance.Brightness(img).enhance(1.1)
    return img

def image_to_bytes(img):
    b = io.BytesIO()
    img.save(b, format="JPEG")
    return b.getvalue()

# ==============================================
# 🔁 Safe Retry Wrapper
# ==============================================
def call_gemini(prompt, img_bytes=None, retries=4):
    for i in range(retries):
        try:
            if img_bytes:
                return model.generate_content([prompt, img_bytes])
            else:
                return model.generate_content(prompt)
        except Exception as e:
            print(f"Retry {i+1}/{retries}: {e}")
            time.sleep(1)
    return None

# ==============================================
# ⭐ Rating Emoji
# ==============================================
def rating_emoji(r):
    return "🤩" if r >= 8 else ("🙂" if r >= 6 else ("😐" if r >= 4 else "🙁"))

# ==============================================
# 🚘 Car + Part Inspection
# ==============================================
def analyze(fullcar, body, wheel1, wheel2, engine):

    html = "<h2 style='color:black'>🚗 Car Inspection Summary</h2>"

    # Helper to process each image
    def process(title, file, prompt):
        if file is None:
            return f"<h3>{title}</h3><p style='color:black;'>❌ Not uploaded.</p>"

        img = Image.fromarray(file)
        img = enhance(img)
        bytes_ = image_to_bytes(img)

        res = call_gemini(prompt, bytes_)
        if not res:
            return f"<h3>{title}</h3><p style='color:black;'>❌ Error contacting Gemini</p>"

        text = res.text if hasattr(res, "text") else str(res)

        # extract rating if exists
        rating_match = re.search(r"(\d+(\.\d+)?)", text)
        rating = float(rating_match.group(1)) if rating_match else 6
        emoji = rating_emoji(rating)

        return f"<h3>{title} {emoji}</h3><p style='color:black;'>{text}</p>"

    # ======= Full Car =======
    html += process(
        "Full Car",
        fullcar,
        """
        Inspect the full car exterior:
        - overall condition
        - major defects
        - dents & scratches
        - colour issues
        - structural problems
        - give rating 1–10
        """
    )

    # ======= Body =======
    html += process(
        "Body",
        body,
        """
        Inspect the car body:
        - rust or corrosion
        - paint quality
        - dent severity
        - structural alignment
        - rating 1–10
        """
    )

    # ======= Wheel 1 =======
    html += process(
        "Wheel 1",
        wheel1,
        """
        Inspect this wheel:
        - tyre wear
        - rim condition
        - cracks, bulges
        - brake visibility
        - rating 1–10
        """
    )

    # ======= Wheel 2 =======
    html += process(
        "Wheel 2",
        wheel2,
        """
        Inspect this wheel:
        - tyre wear pattern
        - alignment issues
        - rim damage
        - rating 1–10
        """
    )

    # ======= Engine =======
    html += process(
        "Engine",
        engine,
        """
        Inspect the engine:
        - leaks
        - rust or corrosion
        - oil condition
        - belts, hoses
        - rating 1–10
        """
    )

    return html

# ==============================================
# 🎨 Blue Background + Black Text CSS
# ==============================================
CSS = """
<style>
body { background: #b3d1ff !important; }
.gradio-container { background: #b3d1ff !important; }
* { color: black !important; }
</style>
"""

# ==============================================
# 🖥 Gradio UI
# ==============================================
with gr.Blocks() as demo:

    gr.HTML(CSS)

    gr.Markdown("<h2 style='text-align:center;color:black;'>🚘 Used Car Multi-Part Inspection</h2>")

    gr.Markdown("### Upload Full Car + Individual Parts")

    with gr.Row():
        fullcar = gr.Image(label="Full Car Image", type="numpy")
        body = gr.Image(label="Body Image", type="numpy")

    with gr.Row():
        wheel1 = gr.Image(label="Wheel 1 Image", type="numpy")
        wheel2 = gr.Image(label="Wheel 2 Image", type="numpy")

    engine = gr.Image(label="Engine Image", type="numpy")

    run_btn = gr.Button("Run Inspection")

    output = gr.HTML()

    run_btn.click(
        analyze,
        [fullcar, body, wheel1, wheel2, engine],
        output
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c1afe2f319dcb19da2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
